**<h1>AHJ Database: City Query**
#### Supporting Analysis
<i>Any question regarding the notebook, please contact Robert Ford<br>
    Last Updated: 10/02/2025</i>
* AHJ Data: City and County level data sourced from Transparent California
* Started development on 10/02/2025
* Finished Iteration on -10/02/2025-

In [55]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [56]:
carlsbad_df = pd.read_csv('C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/AHJ-Database/data/raw/Transparent America Search Results/carlsbad-2024.csv')
chula_df = pd.read_csv('C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/AHJ-Database/data/raw/Transparent America Search Results/chula-vista-2023.csv')
coronado_df = pd.read_csv('C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/AHJ-Database/data/raw/Transparent America Search Results/coronado-2023.csv')
delmar_df = pd.read_csv('C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/AHJ-Database/data/raw/Transparent America Search Results/del-mar-2023.csv')
escondido_df = pd.read_csv('C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/AHJ-Database/data/raw/Transparent America Search Results/escondido-2024.csv')
el_cajon_df = pd.read_csv('C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/AHJ-Database/data/raw/Transparent America Search Results/el-cajon-2023.csv')
encitas_df = pd.read_csv('C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/AHJ-Database/data/raw/Transparent America Search Results/encinitas-2024.csv')
imperial_beach_df = pd.read_csv('C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/AHJ-Database/data/raw/Transparent America Search Results/imperial-beach-2023.csv')
la_mesa_df = pd.read_csv('C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/AHJ-Database/data/raw/Transparent America Search Results/la-mesa-2023.csv')
lemon_grove_df = pd.read_csv('C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/AHJ-Database/data/raw/Transparent America Search Results/lemon-grove-2023.csv')
national_city_df = pd.read_csv('C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/AHJ-Database/data/raw/Transparent America Search Results/national-city-2023.csv')

carlsbad_df['City'] = 'Carlsbad'
chula_df['City'] = 'Chula Vista'
coronado_df['City'] = 'Coronado'
delmar_df['City'] = 'Del Mar'
escondido_df['City'] = 'Escondido'
el_cajon_df['City'] = 'El Cajon'
encitas_df['City'] = 'Encinitas'
imperial_beach_df['City'] = 'Imperial Beach'
la_mesa_df['City'] = 'La Mesa'
lemon_grove_df['City'] = 'Lemon Grove'
national_city_df['City'] = 'National City'


In [57]:
# Combine all dataframes into one
dfs = [
    carlsbad_df, chula_df, coronado_df, delmar_df, escondido_df, el_cajon_df,
    encitas_df, imperial_beach_df, la_mesa_df, lemon_grove_df, national_city_df
]
df = pd.concat(dfs, ignore_index=True)

In [58]:
def clean_job_title(title):
    if pd.isnull(title):
        return ""
    
    title = title.strip().lower()
    title = re.sub(r"(h/|int/|temp/|^pt\s*)", "", title)  # Remove common prefixes
    title = re.sub(r"[^a-zA-Z\s]", "", title)  # Remove special characters
    title = re.sub(r"\s+", " ", title)  # Normalize whitespace
    return title.title()

df['Cleaned_Title'] = df['Job Title'].apply(clean_job_title)


In [59]:
keywords = [
    "inspector", "planner", "planning", "commission", "contractor",
    "construction", "building", "public works", "code enforcement",
    "architect", "community development", "engineer", "permit", "plan check"
]

def is_building_related(title):
    title = title.lower()
    return any(keyword in title for keyword in keywords)

df['Is_BuildingDept'] = df['Cleaned_Title'].apply(is_building_related)

In [60]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['Cleaned_Title'])
y = df['Is_BuildingDept']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [61]:
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

       False       0.99      1.00      0.99      1645
        True       1.00      0.89      0.94       173

    accuracy                           0.99      1818
   macro avg       0.99      0.95      0.97      1818
weighted avg       0.99      0.99      0.99      1818



In [ ]:
# Count estimated building department staff per city
building_staff_counts = df[df['Is_BuildingDept']].groupby('City').size().sort_values(ascending=False)
city_staff_totals = df.groupby('City').size()
building_staff_percentage = (building_staff_counts / city_staff_totals * 100).fillna(0)

print("Estimated Building Department Staffing by City:")
print(building_staff_counts)
print("\nPercentage of Building Department Staff by City:")
print(building_staff_percentage)
print("\nTotal Staff Count by City:")
print(city_staff_totals)


Total Staff Count by City:
City
Carlsbad          1473
Chula Vista       1774
Coronado           596
Del Mar            175
El Cajon           549
Encinitas          452
Escondido         1187
Imperial Beach     178
La Mesa            368
Lemon Grove         55
National City      464
dtype: int64


In [70]:
# Combine building staff counts, total staff, and percentage into one concise table
summary_df = pd.DataFrame({
    'Building Dept Staff': building_staff_counts,
    'Total Staff': city_staff_totals,
    'Building Dept %': building_staff_percentage
}).fillna(0).sort_values('Building Dept Staff', ascending=False)

print(summary_df)

                Building Dept Staff  Total Staff  Building Dept %
City                                                             
Chula Vista                     192         1774        10.822999
Carlsbad                        148         1473        10.047522
Escondido                        91         1187         7.666386
Encinitas                        61          452        13.495575
La Mesa                          47          368        12.771739
National City                    44          464         9.482759
El Cajon                         34          549         6.193078
Coronado                         27          596         4.530201
Del Mar                          20          175        11.428571
Imperial Beach                   15          178         8.426966
Lemon Grove                      12           55        21.818182
